# Mercado Laboral y Salarios en Tecnología (Chile y Latinoamérica 2024–2026)
### Certificado Profesional de Análisis de Datos de Google • Caso Práctico 3 (Pista B)
**Autor:** Manuel Yáñez Orellana  
**Patrocinador Simulado:** Consultora de Recursos Humanos Tech & Academia de Formación Digital (Chile)  
**Sitio Web Oficial (GitHub Pages):** [https://jmryanez.github.io/mercado-tech-chile-latam-2026/](https://jmryanez.github.io/mercado-tech-chile-latam-2026/)  
**Repositorio en GitHub:** [https://github.com/jmryanez/mercado-tech-chile-latam-2026](https://github.com/jmryanez/mercado-tech-chile-latam-2026)  
**Marco Metodológico:** Cumplimiento estricto de las directrices metodológicas de los **9 cursos oficiales para la obtención de la Certificación Profesional de Google Data Analytics**.

---

## Resumen del Caso de Estudio
Este cuaderno reproduce de forma autónoma y determinista el flujo analítico completo (**Fases 1 a 6**) del estudio sobre demanda tecnológica y compensaciones salariales en Chile y América Latina.  
El análisis procesa cuatro fuentes inmutables manteniendo las **4 poblaciones estrictamente separadas** (sin promediar ni sumar juntas bolsas de trabajo con encuestas comunitarias) y aplicando la regla muestral **`N >= 30`** fundamentada en el **Teorema del Límite Central (CLT) al 95% de confianza** (Curso 4, Módulo 1).

In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    display = print

plt.rcParams["text.parse_math"] = False
sns.set_theme(style="whitegrid")

# Autodetección robusta de la ruta de datos (Kaggle Input, Repositorio local o clon directo)
CANDIDATE_DIRS = [
    Path("/kaggle/input/mercado-tech-chile-latam-limpios"),
    Path("/kaggle/input/mercado-tech-chile-latam-2026/03_Fase_Procesar_Process/datos_limpios_processed"),
    Path("/kaggle/input/mercado-tech-chile-latam-2026"),
    Path("03_Fase_Procesar_Process/datos_limpios_processed"),
    Path("../03_Fase_Procesar_Process/datos_limpios_processed"),
    Path("../../03_Fase_Procesar_Process/datos_limpios_processed"),
    Path.cwd() / "datos_limpios_processed",
    Path.cwd(),
]

DATA_DIR = next(
    (d for d in CANDIDATE_DIRS if (d / "GetOnBoard_Vacantes_2026_20261111_v01_clean.csv").exists()),
    Path("03_Fase_Procesar_Process/datos_limpios_processed")
)
print("Directorio activo de datos limpios:", DATA_DIR.resolve())

In [ ]:
# Ingesta de las 4 poblaciones auditadas (Fase 2: Preparar • Cursos 3 y 4)
gob_vac = pd.read_csv(DATA_DIR / "GetOnBoard_Vacantes_2026_20261111_v01_clean.csv")
gob_hab = pd.read_csv(DATA_DIR / "GetOnBoard_Habilidades_2026_20261111_v01_clean.csv")
so_lat  = pd.read_csv(DATA_DIR / "StackOverflow_LatAm_2024_20261111_v01_clean.csv")
so_tec  = pd.read_csv(DATA_DIR / "StackOverflow_Tecnologias_2024_20261111_v01_clean.csv")
dss     = pd.read_csv(DATA_DIR / "DataScienceSalaries_2020_2024_20261111_v01_clean.csv")
ine     = pd.read_csv(DATA_DIR / "INE_ESI_2025_ReferenciaNacional_20261111_v01_clean.csv")

print("=== INVENTARIO DE POBLACIONES CARGADAS ===")
print(f"1. Get on Board 2026 (API v0): {len(gob_vac)} ofertas (salarios válidos={int(gob_vac['salario_valido_flag'].sum())}, habilidades={len(gob_hab)})")
print(f"2. Stack Overflow LatAm 2024: {len(so_lat)} desarrolladores (salarios válidos={int(so_lat['salario_valido_flag'].sum())}, Chile={int((so_lat['pais']=='Chile').sum())})")
print(f"3. Data Science Salaries (2020–2024): {len(dss)} registros internacionales de compensación")
mediana_ine = int(ine.loc[ine['indicador'].str.contains('Mediano', case=False), 'monto_clp_mensual'].iloc[0])
print(f"4. INE ESI 2025 (Chile): {len(ine)} microdatos | Mediana nacional formal = ${mediana_ine:,} CLP (~$701 USD)")

## 1. Arquitectura Relacional en SQLite (Fase 4 • Curso 5)
Carga de las tablas en una base de datos relacional SQLite en memoria para consultar las tablas normalizadas (Ancho + Largo).
Se audita y controla la **integridad relacional contra la duplicación en uniones 1:N** mediante `LEFT JOIN` y `COUNT(DISTINCT)`, lo que evitó una sobreestimación artificial de **+$75 USD** en el salario medio general.

In [ ]:
conn = sqlite3.connect(":memory:")
gob_vac.to_sql("vacantes", conn, index=False)
gob_hab.to_sql("vacante_habilidades", conn, index=False)
so_lat.to_sql("so_encuestados", conn, index=False)
so_tec.to_sql("so_tecnologias", conn, index=False)

# Consulta Q01 & Q03: Tecnologías más demandadas en avisos de contratación con filtro HAVING >= 20%
q_top5_gob = """
SELECT
    h.tipo_tecnologia,
    h.tecnologia,
    COUNT(DISTINCT h.job_id) AS n_ofertas,
    ROUND(100.0 * COUNT(DISTINCT h.job_id) / (SELECT COUNT(DISTINCT job_id) FROM vacantes), 2) AS pct_demanda
FROM vacante_habilidades AS h
GROUP BY h.tipo_tecnologia, h.tecnologia
HAVING 100.0 * COUNT(DISTINCT h.job_id) / (SELECT COUNT(DISTINCT job_id) FROM vacantes) >= 20.0
ORDER BY pct_demanda DESC;
"""
df_top5 = pd.read_sql_query(q_top5_gob, conn)
display(df_top5)

## 2. Pregunta de Investigación 1 (P1): Demanda vs. Adopción Tecnológica
**Pregunta de Negocio (P1):** *¿Cuáles son las tecnologías más demandadas por los empleadores tech en Chile y qué brechas existen frente a las habilidades que realmente dominan los profesionales?*

**Hallazgo Clave:** Solo 5 tecnologías superan el umbral crítico del 20% en Get on Board (SQL 32,8%, AWS 30,0%, Python 28,4%, JavaScript 27,4% y Azure 20,8%). Existe un déficit crítico en Cloud: **Microsoft Azure** presenta una brecha de demanda no cubierta (+5,7% más solicitada en avisos que dominada por desarrolladores en Chile).

In [ ]:
# Visualización P1: Gráfico de barras horizontal con paleta apta para daltonismo y honestidad escalar (eje en 0%)
fig, ax = plt.subplots(figsize=(9, 4.5), dpi=140)
df_plot = df_top5.sort_values("pct_demanda", ascending=True)
colores = ["#D55E00" if t == "Microsoft Azure" else "#1A73E8" for t in df_plot["tecnologia"]]
bars = ax.barh(df_plot["tecnologia"], df_plot["pct_demanda"], color=colores, height=0.55)

ax.axvline(20.0, color="#D55E00", linestyle="--", linewidth=1.2, label="Meta Crítica >= 20%")
for idx, r in enumerate(df_plot.itertuples()):
    ax.text(r.pct_demanda + 0.6, idx, f"{r.pct_demanda:.2f}% (N={r.n_ofertas})", va="center", fontsize=9, fontweight="bold", color="#1E293B")

ax.set_xlim(0, 42)
ax.set_xlabel("% de Aparición en Vacantes Vigentes en Get on Board (N=679)", fontsize=10, fontweight="bold")
ax.set_title("P1: Solo 5 Tecnologías Superan el Umbral Crítico del >= 20% en Chile/LatAm (2026)", loc="left", fontsize=12, fontweight="bold")
ax.legend(loc="lower right", frameon=True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 3. Pregunta de Investigación 2 (P2): Progresión Salarial y Salto Senior
**Pregunta de Negocio (P2):** *¿Cuál es la estructura salarial por nivel de experiencia y en qué tramo ocurre el mayor salto porcentual?*

**Hallazgo Clave:** Dado el sesgo a la derecha en los ingresos tech, se priorizan Medianas y Rangos Intercuartiles $p25–p75$ (Curso 4). En Chile, el salto más pronunciado de la carrera ocurre en la transición de **Semi-Senior a Senior con un +42,2%** (de $2.082 USD a $2.960 USD).

In [ ]:
# Cálculo de estadísticas robustas no paramétricas por nivel de experiencia
gob_sal = gob_vac[gob_vac["salario_valido_flag"] == True].copy()
bandas_exp = (
    gob_sal.groupby("seniority_nivel")["salario_mensual_usd"]
    .agg(
        n="count",
        mediana_usd="median",
        p25_usd=lambda s: s.quantile(0.25),
        p75_usd=lambda s: s.quantile(0.75),
    )
    .reset_index()
)
bandas_exp["cumple_n30"] = bandas_exp["n"] >= 30
bandas_exp = bandas_exp.sort_values("mediana_usd", ascending=False)
display(bandas_exp)

## 4. Pregunta de Investigación 3 (P3): Modalidad Laboral & Desmitificación de Causalidad
**Pregunta de Negocio (P3):** *¿Existe una prima salarial estadísticamente atribuible al trabajo remoto frente al presencial en Chile?*

**Auditoría de Causalidad (Q12):** La regla de oro de Google establece que **Correlación no implica Causalidad**. Aunque el esquema Remoto Local aparenta pagar un 18% más que el Presencial, la tabla cruzada demuestra que el 78% de los puestos remotos corresponden a cargos Senior y Lead. Al controlar por seniority, la prima se neutraliza.

In [ ]:
# Validación de Composición Q12: Modalidad x Seniority
comp_q12 = (
    gob_sal.groupby(["modalidad_grupo", "seniority_nivel"])
    .size()
    .unstack(fill_value=0)
)
print("=== COMPOSICIÓN MODALIDAD X SENIORITY (GOB) ===")
display(comp_q12)

## 5. Pregunta de Investigación 4 (P4) & Matriz TLC de 19 Países de LatAm
**Pregunta de Negocio (P4):** *¿Cómo se posiciona Chile en remuneraciones respecto al resto de Latinoamérica bajo suficiencia muestral N >= 30?*

**Hallazgo Clave:** De los 19 países de Latinoamérica encuestados en Stack Overflow 2024, solo 7 cumplen con $N \ge 30$. Chile ocupa la 2ª posición regional con una mediana de **$3.063 USD/mes**, superado únicamente por Uruguay ($6.416 USD/mes), y por encima de Argentina ($2.958 USD), México ($2.632 USD) y Brasil ($2.020 USD).

In [ ]:
# Carga y despliegue de la Matriz de Representatividad TLC de 19 Países (N >= 30)
candidate_tlc = [
    Path("descargas/Matriz_TLC_19_Paises_LatAm.csv"),
    Path("../descargas/Matriz_TLC_19_Paises_LatAm.csv"),
    DATA_DIR / "Matriz_TLC_19_Paises_LatAm.csv",
    Path("/kaggle/input/mercado-tech-chile-latam-limpios/Matriz_TLC_19_Paises_LatAm.csv"),
    Path("/kaggle/input/mercado-tech-chile-latam-2026/descargas/Matriz_TLC_19_Paises_LatAm.csv"),
]
tlc_file = next((f for f in candidate_tlc if f.exists()), candidate_tlc[0])
tlc_df = pd.read_csv(tlc_file)
print(f"=== MATRIZ TLC DE 19 PAÍSES DE LATINOAMÉRICA (Cargado desde: {tlc_file}) ===")
display(tlc_df[["pais", "n_total", "n_sal", "cumple_n30", "med_usd", "error_est", "tratamiento"]].head(10))


## 6. Plan de Acción Accionable y Recomendaciones (Fase 6 • Curso 8)
Basado en los hallazgos con representatividad estadística ($N \ge 30$), se establecen tres recomendaciones prioritarias con matriz de implementación Qué / Quién / Cuándo:

| # | Acción Concreta (Qué) | Responsable (Quién) | Plazo (Cuándo) | Métrica de Éxito (KPI) |
| :---: | :--- | :--- | :---: | :--- |
| **1** | **Rediseño Curricular Acelerado SQL + Cloud:** Incorporar módulos intensivos de SQL relacional y certificación inicial en Cloud (AWS o Azure) en todos los programas de la academia. | Dirección Académica & Coordinación Tech | **30 Días** | 100% de los egresados aprueban evaluación técnica en SQL y Cloud básico. |
| **2** | **Puente Formativo Semi-Senior a Senior:** Crear programas de mentoría técnica y proyectos con datos masivos para acortar la brecha de transición salarial (+42,2%). | Líderes Técnicos & Mentores Senior | **60 Días** | Reducción de 24 a 14 meses en el tiempo promedio de ascenso a nivel Senior. |
| **3** | **Benchmarking Salarial & Transparencia:** Asesorar a empresas clientes para publicar rangos salariales explícitos y competitivos según las bandas p25–p75 observadas. | Gerencia de Consultoría RRHH Tech | **90 Días** | Reducción del 35% en el tiempo de cobertura de vacantes críticas (Time-to-Hire). |

---

## 7. Kit de Carrera y Marca Profesional (Curso 9)
Herramientas de empleabilidad desarrolladas en el módulo final del Certificado Profesional de Google:

* **Metodología STAR (Situación, Tarea, Acción, Resultado):** Estructuración de la narrativa técnica para entrevistas de trabajo.
* **Fórmula X-Y-Z de Google para Currículum:** *«Logró [X], medido por [Y], haciendo [Z]»*:
  * *«Lideró el análisis de 72.797 registros laborales tech en Chile y LatAm, cuantificando un salto remuneracional de +42,2% al nivel Senior y una brecha de demanda no cubierta de +5,7% en Azure, implementando un pipeline determinista en Python y SQLite relacional.»*
* **Discurso de Presentación Profesional (Elevator Pitch de 20 Segundos):** Síntesis ejecutiva de alto impacto para comités de contratación y líderes de negocio.